# 발화 기록 노트 — 목소리 복제 서버 (무료 · 구글 코랩)

학생 목소리를 복제해 한국어 문장을 읽어 주는 서버를 코랩 무료 GPU 로 띄웁니다. 비용은 0원이고 카드 등록도 필요 없습니다.

## 쓰는 순서

1. 위쪽 메뉴에서 런타임 → 런타임 유형 변경 → 하드웨어 가속기를 **T4 GPU** 로 바꾸고 저장합니다.
2. 아래 1번 칸을 실행합니다. 5~10분쯤 걸립니다. (처음 한 번만)
3. 2번 칸의 공유 암호(APP_TOKEN)를 **영문·숫자**로 된 아무 문자열로 바꾸고 실행합니다. 한글은 통신 규약상 쓸 수 없습니다.
4. 마지막에 나오는 주소와 공유 암호를 앱 설정의 목소리 복제 항목에 넣습니다.
5. 수업이 끝나면 이 탭을 닫으시면 됩니다.

## 알아 두실 점

- 코랩 무료는 한 번에 최대 12시간, 90분쯤 손대지 않으면 끊깁니다. 끊기면 2번 칸만 다시 실행하고 **새 주소를 앱에 다시 넣으십시오**. 주소는 실행할 때마다 바뀝니다.
- 첫 문장은 모델을 올리느라 20~40초, 그다음부터는 3~8초쯤 걸립니다.
- XTTS-v2 모델은 CPML(비상업) 라이선스입니다. 교육·연구 용도에 적합합니다.
- 학생 음성은 코랩(구글) 서버로 전송됩니다. 보호자 동의서에 국외 이전 항목이 있는지 확인하십시오.


In [ ]:
#@title 1. 설치 (처음 한 번만, 5~10분)
import os, subprocess, sys

os.environ["COQUI_TOS_AGREED"] = "1"

print("설치를 시작합니다. 5~10분쯤 걸립니다.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "coqui-tts", "fastapi", "uvicorn[standard]", "python-multipart"], check=False)

# 터널 프로그램 (앱이 접속할 https 주소를 만들어 줍니다)
subprocess.run("wget -q -O cloudflared.deb "
               "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
               shell=True, check=False)
subprocess.run("dpkg -i cloudflared.deb > /dev/null 2>&1", shell=True, check=False)

import torch
print("설치 끝. GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(런타임 유형을 T4 GPU 로 바꾸세요)")


In [ ]:
#@title 2. 서버 켜기 (수업할 때마다 실행)

APP_TOKEN = "change-me-2026-abcd1234" #@param {type:"string"}
ALLOW_ORIGIN = "https://86loading-droid.github.io" #@param {type:"string"}
MAX_CHARS = 200 #@param {type:"integer"}

import os, re, io, time, uuid, threading, subprocess, tempfile
os.environ["COQUI_TOS_AGREED"] = "1"

# 공유 암호는 HTTP 헤더로 오갑니다. 헤더는 한글을 담지 못하므로 영문·숫자만 받습니다.
try:
    APP_TOKEN.encode("latin-1")
except UnicodeEncodeError:
    raise SystemExit("공유 암호(APP_TOKEN)에 한글이 들어 있습니다. "
                     "영문·숫자·기호로만 바꾸신 뒤(예: kse-voice-2026-9f3a) 이 칸을 다시 실행하십시오.")

import torch
# XTTS 는 예전 방식으로 저장된 파일을 읽습니다. 최신 torch 에서 나는 오류를 막습니다.
# 이 칸을 다시 실행해도 덧씌워지지 않도록 한 번만 감쌉니다.
if not getattr(torch.load, "_speechlog_patched", False):
    _orig_load = torch.load
    def _patched_load(*a, **k):
        k["weights_only"] = False
        return _orig_load(*a, **k)
    _patched_load._speechlog_patched = True
    torch.load = _patched_load

from fastapi import FastAPI, UploadFile, File, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response
import uvicorn

MODEL = "tts_models/multilingual/multi-dataset/xtts_v2"
# 이 칸을 다시 실행해도 이미 올린 모델과 등록한 목소리는 그대로 둡니다.
VOICES = globals().get("VOICES", {})   # 등록한 목소리 {번호: 참조 음성 경로}
_tts = globals().get("_tts", None)
# 런타임을 새로 켤 때마다 바뀌는 번호. 앱은 이 번호로 사라진 등록을 알아채고 정리합니다.
BOOT = globals().get("BOOT") or uuid.uuid4().hex[:8]
_lock = threading.Lock()

def _fix_transformers():
    """coqui-tts 가 찾는 함수가 최신 transformers 에 없을 때 채워 넣는다."""
    try:
        import transformers.pytorch_utils as tpu
        if not hasattr(tpu, "isin_mps_friendly"):
            tpu.isin_mps_friendly = lambda elements, test_elements: torch.isin(elements, test_elements)
    except Exception as e:
        print("transformers 보정 건너뜀:", e)

def get_tts():
    global _tts
    if _tts is None:
        _fix_transformers()
        import sys
        for _m in [m for m in list(sys.modules) if m.startswith("TTS")]:
            sys.modules.pop(_m, None)
        from TTS.api import TTS
        print("모델을 불러옵니다. 처음에는 몇 분 걸립니다…")
        t = TTS(MODEL)
        t.to("cuda" if torch.cuda.is_available() else "cpu")
        _tts = t
        print("모델 준비 끝.")
    return _tts

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=[ALLOW_ORIGIN] if ALLOW_ORIGIN else ["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Content-Type", "X-App-Token"],
)

def bad(msg, code=400):
    return JSONResponse({"error": msg}, status_code=code)

def ok_token(req: Request):
    if not APP_TOKEN:
        return True
    return req.headers.get("x-app-token") == APP_TOKEN

@app.get("/health")
def health():
    return {"ok": True, "provider": "colab/xtts-v2", "backend": "colab",
            "keyed": True, "maxChars": MAX_CHARS, "free": True, "boot": BOOT}

@app.post("/voice")
async def voice(request: Request, file: UploadFile = File(...)):
    if not ok_token(request):
        return bad("인증 실패: 공유 암호가 맞지 않습니다.", 401)
    raw = await file.read()
    if not raw:
        return bad("참조 음성이 없습니다.")
    src = tempfile.NamedTemporaryFile(suffix=".webm", delete=False).name
    with open(src, "wb") as f:
        f.write(raw)
    dst = src + ".wav"
    subprocess.run(["ffmpeg", "-y", "-i", src, "-ar", "22050", "-ac", "1", dst],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not os.path.exists(dst) or os.path.getsize(dst) < 1000:
        return bad("참조 음성을 읽지 못했습니다. 조금 더 길게 녹음해 주세요.", 400)
    vid = uuid.uuid4().hex
    VOICES[vid] = dst
    print("목소리 등록:", vid, os.path.getsize(dst), "바이트")
    return {"voiceUrl": vid, "boot": BOOT}

@app.post("/speak")
async def speak(request: Request):
    if not ok_token(request):
        return bad("인증 실패: 공유 암호가 맞지 않습니다.", 401)
    body = await request.json()
    vid = str(body.get("voiceUrl") or "").strip()
    text = str(body.get("text") or "").strip()
    if vid not in VOICES:
        return bad("등록된 목소리가 없습니다. 설정에서 다시 등록해 주세요.")
    if not text:
        return bad("읽을 내용이 없습니다.")
    if len(text) > MAX_CHARS:
        return bad(f"한 번에 {MAX_CHARS}자까지만 읽어 줍니다.", 413)
    out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    with _lock:
        t0 = time.time()
        get_tts().tts_to_file(text=text, speaker_wav=VOICES[vid], language="ko", file_path=out)
        print(f"읽기 완료 {time.time()-t0:.1f}초 · {len(text)}자")
    with open(out, "rb") as f:
        data = f.read()
    return Response(content=data, media_type="audio/wav")

# 앞서 실행한 터널이 남아 있으면 정리합니다.
subprocess.run("pkill -f cloudflared", shell=True, check=False)

# 앞선 실행이 쓰던 문(포트)을 피해 빈 자리를 찾습니다.
import socket
_s = socket.socket()
_s.bind(("127.0.0.1", 0))
PORT = _s.getsockname()[1]
_s.close()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

# 터널을 열어 https 주소를 만든다
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = ""
t_end = time.time() + 90
while time.time() < t_end:
    line = proc.stdout.readline()
    if not line:
        continue
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n" + "=" * 60)
if url:
    print("앱 설정에 아래 두 가지를 넣으십시오.")
    print()
    print("  복제 서버 주소 :", url)
    print("  공유 암호      :", APP_TOKEN)
    print()
    print("이 칸을 멈추면 서버도 꺼집니다. 수업 동안 켜 두십시오.")
    print("다시 실행하면 주소가 바뀌므로 앱에 새 주소를 넣으셔야 합니다.")
else:
    print("주소를 만들지 못했습니다. 이 칸을 다시 실행해 주십시오.")
print("=" * 60)

# 모델을 미리 올려 첫 문장을 빠르게 한다
try:
    get_tts()
except Exception as e:
    print("모델 준비 중 오류:", e)

# 칸이 끝나지 않도록 붙잡아 둔다
while True:
    time.sleep(60)
